In [2]:
!pip install transformers datasets evaluate torch torchvision scikit-learn matplotlib seaborn pillow -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
TRAIN_DIR = "../data/face/RAF-DB/DATASET/train"
TEST_DIR  = "../data/face/RAF-DB/DATASET/test"

In [3]:
# RAF folders assumed:
# 1=Surprise, 2=Fear, 3=Disgust, 4=Happy, 5=Sad, 6=Anger, 7=Neutral

raf_to_final = {
    "1": "Neutral",      # Surprise
    "2": "Anxiety",      # Fear
    "3": "Negative",     # Disgust
    "4": "Positive",     # Happy
    "5": "Depression",   # Sad
    "6": "Stress",       # Anger
    "7": "Neutral"       # Neutral
}

class_names = [
    "Positive",
    "Neutral",
    "Stress",
    "Anxiety",
    "Negative",
    "Depression"
]

label2id = {name:i for i,name in enumerate(class_names)}
id2label = {i:name for name,i in label2id.items()}

In [4]:
def load_folder(data_dir):
    images = []
    labels = []

    for folder in os.listdir(data_dir):
        folder_path = os.path.join(data_dir, folder)

        if not os.path.isdir(folder_path):
            continue

        final_label = raf_to_final.get(folder)
        if final_label is None:
            continue

        for file in os.listdir(folder_path):
            path = os.path.join(folder_path, file)
            images.append(path)
            labels.append(label2id[final_label])

    return pd.DataFrame({
        "image": images,
        "label": labels
    })

In [5]:
train_df = load_folder(TRAIN_DIR)
test_df = load_folder(TEST_DIR)

print(train_df.head())
print("Train:", len(train_df))
print("Test :", len(test_df))

                                               image  label
0  ../data/face/RAF-DB/DATASET/train\1\train_0000...      1
1  ../data/face/RAF-DB/DATASET/train\1\train_0001...      1
2  ../data/face/RAF-DB/DATASET/train\1\train_0001...      1
3  ../data/face/RAF-DB/DATASET/train\1\train_0001...      1
4  ../data/face/RAF-DB/DATASET/train\1\train_0003...      1
Train: 12271
Test : 3068


In [6]:
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

In [7]:
model_name = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(model_name)

In [8]:
def transform(example):
    image = Image.open(example["image"]).convert("RGB")

    encoded = processor(
        images=image,
        return_tensors="pt"
    )

    example["pixel_values"] = encoded["pixel_values"][0]
    return example

In [9]:
train_ds = train_ds.map(transform)
test_ds = test_ds.map(transform)

Map:   0%|          | 0/12271 [00:00<?, ? examples/s]

Map:   0%|          | 0/3068 [00:00<?, ? examples/s]

In [10]:
train_ds.set_format(type="torch", columns=["pixel_values", "label"])
test_ds.set_format(type="torch", columns=["pixel_values", "label"])

In [11]:
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=6,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

You passed `num_labels=6` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([6])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([6, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [13]:
args = TrainingArguments(
    output_dir="../models/face_final_6",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50
)

In [14]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

c:\Users\prasa\OneDrive\Desktop\BE\backend\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
pred_output = trainer.predict(test_ds)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Face Model Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

metrics_table = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Score": [
        round(accuracy,4),
        round(precision,4),
        round(recall,4),
        round(f1,4)
    ]
})

metrics_table

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

In [ ]:
model.save_pretrained("../models/face_final_6")
processor.save_pretrained("../models/face_final_6")

In [ ]:
from transformers import pipeline

clf = pipeline(
    "image-classification",
    model="../models/face_final_6",
    image_processor="../models/face_final_6"
)

img_path = train_df.iloc[0]["image"]
print(clf(img_path))